In [1]:
import matplotlib.pyplot as plt
import numpy as np
from dotenv import load_dotenv
import ocha_stratus as stratus
from src.datasources import glofas
from src.utils import blob

load_dotenv()

/Users/hannahker/Desktop/AA/ds-aa-nga-flooding/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
STATION_NAME = "makurdi"

## Check raw blobs

In [3]:
raw_prefix = f"ds-aa-nga-flooding/raw/glofas/reanalysis/glofas_raw_reanalysis_{STATION_NAME}"
raw_blobs = sorted(
    x for x in blob.list_container_blobs(name_starts_with=raw_prefix)
    if x.endswith(".grib")
)
print(f"Found {len(raw_blobs)} raw GRIB files")
for b in raw_blobs:
    print(" ", b)

Found 46 raw GRIB files
  ds-aa-nga-flooding/raw/glofas/reanalysis/glofas_raw_reanalysis_makurdi_1979.grib
  ds-aa-nga-flooding/raw/glofas/reanalysis/glofas_raw_reanalysis_makurdi_1980.grib
  ds-aa-nga-flooding/raw/glofas/reanalysis/glofas_raw_reanalysis_makurdi_1981.grib
  ds-aa-nga-flooding/raw/glofas/reanalysis/glofas_raw_reanalysis_makurdi_1982.grib
  ds-aa-nga-flooding/raw/glofas/reanalysis/glofas_raw_reanalysis_makurdi_1983.grib
  ds-aa-nga-flooding/raw/glofas/reanalysis/glofas_raw_reanalysis_makurdi_1984.grib
  ds-aa-nga-flooding/raw/glofas/reanalysis/glofas_raw_reanalysis_makurdi_1985.grib
  ds-aa-nga-flooding/raw/glofas/reanalysis/glofas_raw_reanalysis_makurdi_1986.grib
  ds-aa-nga-flooding/raw/glofas/reanalysis/glofas_raw_reanalysis_makurdi_1987.grib
  ds-aa-nga-flooding/raw/glofas/reanalysis/glofas_raw_reanalysis_makurdi_1988.grib
  ds-aa-nga-flooding/raw/glofas/reanalysis/glofas_raw_reanalysis_makurdi_1989.grib
  ds-aa-nga-flooding/raw/glofas/reanalysis/glofas_raw_reanalysi

## Process and upload

Reads each GRIB, extracts `dis24` at the station grid point, concatenates all years, and uploads to:
`ds-aa-nga-flooding/processed/glofas/glofas_reanalysis_{station}.parquet`

## Inspect raw dataset

In [4]:
# Load a single raw GRIB to inspect grid structure
ds_raw = glofas.load_glofas_reanalysis_year("raw", STATION_NAME, 1979)
print(ds_raw)
print("\ndims:", dict(ds_raw.dims))
print("coords:", list(ds_raw.coords))

# Show what to_dataframe() produces before any selection
df_full = ds_raw["dis24"].to_dataframe().reset_index()
print(f"\nto_dataframe() shape: {df_full.shape}")
print(f"columns: {list(df_full.columns)}")
print(df_full.head(10))

<xarray.Dataset> Size: 4kB
Dimensions:     (time: 214, latitude: 1, longitude: 1)
Coordinates:
  * time        (time) datetime64[ns] 2kB 1979-06-01 1979-06-02 ... 1979-12-31
    valid_time  (time) datetime64[ns] 2kB 1979-06-02 1979-06-03 ... 1980-01-01
  * latitude    (latitude) float64 8B 7.775
  * longitude   (longitude) float64 8B 8.525
    step        timedelta64[ns] 8B 1 days
    surface     float64 8B 0.0
Data variables:
    dis24       (time, latitude, longitude) float32 856B 3.584e+03 ... 274.1
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-05-25T11:41 GRIB to CDM+CF via cfgrib-0.9.1...

dims: {'time': 214, 'latitude': 1, 'longitude': 1}
coords: ['time', 'step', 'surface', 'latitude', 'longit

/var/folders/rv/xmclt0vn5y7cqt46s5xq3h080000gn/T/ipykernel_86972/2347505647.py:4: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("\ndims:", dict(ds_raw.dims))


In [5]:
glofas.process_glofas_reanalysis(STATION_NAME)

100%|██████████| 46/46 [00:02<00:00, 20.81it/s]


## Verify output

In [6]:
df = glofas.load_glofas_reanalysis(STATION_NAME)
print(f"Rows: {len(df):,}")
print(f"Date range: {df['time'].min().date()} → {df['time'].max().date()}")
df.tail()

Rows: 9,844
Date range: 1979-06-01 → 2024-12-31


,time,dis24
9839,2024-12-27,143.492188
9840,2024-12-28,138.843750
9841,2024-12-29,134.898438
9842,2024-12-30,131.507812
9843,2024-12-31,129.007812


In [7]:
glofas.load_glofas_reanalysis("wuroboki")

,time,dis24
0,1979-01-01,79.656250
1,1979-01-02,77.609375
2,1979-01-03,76.515625
3,1979-01-04,75.671875
4,1979-01-05,74.843750
...,...,...
16736,2024-10-27,1411.625000
16737,2024-10-28,1280.859375
16738,2024-10-29,1156.441406
16739,2024-10-30,1043.812500


In [8]:
df

,time,dis24
0,1979-06-01,3584.468750
1,1979-06-02,3853.062500
2,1979-06-03,3965.625000
3,1979-06-04,3832.000000
4,1979-06-05,3572.093750
...,...,...
9839,2024-12-27,143.492188
9840,2024-12-28,138.843750
9841,2024-12-29,134.898438
9842,2024-12-30,131.507812
